In [1]:
#confusion_matrix_dtop_weight = {0:-1}
#params_dtop_weight = {0:-1}
%store -r
beta2_params_dt_fcw_op = {'one': {-1}}
beta2_matrix_dt_fcw_op = {'one': {-1}}
beta2_scores_dt_fcw_op = {'one': {'fbeta0':-1, 'recall0':-1, 'precision0':-1, 'accuracy':-1, 'fbeta1':-1, 'recall1':-1,
'precision1':-1}}

#%store -r

In [2]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, fbeta_score, recall_score, precision_score
#from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
#from pprint import pprint
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score


df = pd.read_csv("term-deposit-marketing-2020.csv")

df.replace("unknown", np.nan, inplace=True)

df["y"] = df["y"].map({"no": 0, "yes": 1})

binary_cols = ["housing", "loan", "default"]

for col in binary_cols:
    df[col] = df[col].map({"no": 0, "yes": 1})

df["education"] = df["education"].map({"primary": 1, "secondary": 2, "tertiary": 3})

/data/Apziva/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
drop_cols = [
    "contact",
    "duration",
    "campaign",
    "month",
    "day"
]

df = df.drop(columns=drop_cols)

In [4]:
df = pd.get_dummies(
    df,
    columns=["job", "marital"],
    drop_first=True,
    dtype=int
)

In [5]:
X = df.drop("y", axis=1)
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1234,
    stratify=y
)

In [6]:
##!
##!
beta2 = {'four':2.0, 'qrtr':0.5, 'one':1.0, 'half':0.707, 'two':1.414, 'three':1.73, 'third':0.577, 'zero':0.0, 'inf':10000, 'ten':3.16,
        'five':2.23, '4.5':2.12, '4.25':2.06}

strvar = 'one'
#weight_list = ['balanced', None, {0:1, 1:2}, {0:1, 1:3}, {0:1, 1:4}, {0:1, 1:5}, {0:1, 1:6}]

beta_scorer = make_scorer(fbeta_score, beta=beta2[strvar], pos_label=1)
#beta_scorer = make_scorer(recall_score, pos_label=1)


In [7]:
cw_map = {
      1: None, #bad
      0: "balanced", #better
      2: {0: 1, 1: 2}, #not great
      3: {0: 1, 1: 3}, #not great
      4: {0: 1, 1: 4}, #not great
      5: {0: 1, 1: 5}, #bad
      6: {0: 1, 1: 6},
      7: {0: 1, 1: 7},
      8: {0: 1, 1: 8},
      9: {0: 1, 1: 9},
      10: {0: 1, 1: 10},
      11: {0: 1, 1: 11},
      12: {0: 1, 1: 12},
      13: {0: 1, 1: 13},
      14: {0: 1, 1: 14},
      20: {0: 1, 1: 20}, #great
      21: {0: 1, 1: 21},
      16: {0: 1, 1: 16},
      15: {0: 1, 1: 15},
      22: {0: 1, 1: 22},
      23: {0: 1, 1: 23},
      24: {0: 1, 1: 24},
      25: {0: 1, 1: 25},
      26: {0: 1, 1: 26},
      27: {0: 1, 1: 27},
      28: {0: 1, 1: 28},
      29: {0: 1, 1: 29},
      30: {0: 1, 1: 30},
      19: {0: 1, 1: 19},
      18: {0: 1, 1: 18},
      17: {0: 1, 1: 17},
      31: {0: 1, 1: 31},
}

In [8]:
num = 31  #not done 3-15
def objective(trial):

    cw_opt = trial.suggest_categorical("class_weight_opt", [num])#[0, 1, 2, 3, 4, 5])

    model = DecisionTreeClassifier(
        ccp_alpha=trial.suggest_float('ccp_alpha',0.0, 0.02),
        max_depth=trial.suggest_int("max_depth", 2, 30), #3-15 / 2-30
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20), #2-20 /
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 15), #1-10 /1-20
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        #class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),#,{0:1, 1:2}, {0:1, 1:3}]),#, {0:1, 1:4}, {0:2, 1:1}]),
        class_weight=cw_map[cw_opt],
        splitter=trial.suggest_categorical('splitter', ['best', 'random']),
        criterion=trial.suggest_categorical("criterion", ['gini', 'entropy', 'log_loss']),
        #class_weight=['balanced'],
        random_state=1234
        
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=2,
        scoring=beta_scorer
    )

    return scores.mean()

In [9]:
sampler = optuna.samplers.TPESampler(seed=1234)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=sampler)

In [10]:
study.optimize(objective, n_trials=250)

In [11]:
print(study.best_params)
print(study.best_value)

{'class_weight_opt': 31, 'ccp_alpha': 0.00030320959037639743, 'max_depth': 28, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2', 'splitter': 'best', 'criterion': 'log_loss'}
0.15511110050359198


In [12]:
best = study.best_params
cw_opt = best.pop("class_weight_opt")

best_model = DecisionTreeClassifier(
    random_state=1234,
    class_weight=cw_map[cw_opt],
    #**study.best_params
    **best
)

best_model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'log_loss'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",28
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",4
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",'log2'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",1234
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the c

In [13]:
#best_model.tree_.node_count

In [14]:
#print(best_model.get_depth())
#print(best_model.get_n_leaves())

In [15]:
#print(best_model.tree_.weighted_n_node_samples)

In [16]:
#print(best_model.class_weight)

In [17]:
pred = best_model.predict(X_test)

In [18]:
print("this is the confustion matrix:\n", confusion_matrix(y_test, pred))
print("classification report:\n", classification_report(y_test, pred))
print("this is the weight", cw_map[num])

this is the confustion matrix:
 [[2010 5411]
 [ 120  459]]
classification report:
               precision    recall  f1-score   support

           0       0.94      0.27      0.42      7421
           1       0.08      0.79      0.14       579

    accuracy                           0.31      8000
   macro avg       0.51      0.53      0.28      8000
weighted avg       0.88      0.31      0.40      8000

this is the weight {0: 1, 1: 31}


In [19]:
confusion_matrix_dtop_weight[num]=confusion_matrix(y_test, pred)

params_dtop_weight[num]=study.best_params

for weight, matrix in sorted(confusion_matrix_dtop_weight.items()):
    print(f"{weight}:\n {matrix}")

print("\n")

for weight, params in sorted(params_dtop_weight.items()):
    print(f"{weight}:\n {params}")

print("\n")

%store confusion_matrix_dtop_weight
%store params_dtop_weight

0:
 [[4032 3389]
 [ 239  340]]
1:
 [[6957  464]
 [ 494   85]]
2:
 [[7034  387]
 [ 510   69]]
3:
 [[7233  188]
 [ 517   62]]
4:
 [[6971  450]
 [ 502   77]]
5:
 [[6917  504]
 [ 481   98]]
6:
 [[7012  409]
 [ 492   87]]
7:
 [[7030  391]
 [ 488   91]]
8:
 [[6998  423]
 [ 480   99]]
9:
 [[6857  564]
 [ 463  116]]
10:
 [[6443  978]
 [ 419  160]]
11:
 [[6065 1356]
 [ 378  201]]
12:
 [[4701 2720]
 [ 274  305]]
13:
 [[4724 2697]
 [ 270  309]]
14:
 [[4531 2890]
 [ 256  323]]
15:
 [[4545 2876]
 [ 278  301]]
16:
 [[1743 5678]
 [  68  511]]
17:
 [[   0 7421]
 [   0  579]]
18:
 [[ 559 6862]
 [  21  558]]
19:
 [[2135 5286]
 [  88  491]]
20:
 [[1059 6362]
 [  49  530]]
21:
 [[   0 7421]
 [   0  579]]
22:
 [[1060 6361]
 [  46  533]]
23:
 [[1540 5881]
 [  68  511]]
24:
 [[1838 5583]
 [  89  490]]
25:
 [[ 907 6514]
 [  40  539]]
26:
 [[1963 5458]
 [ 124  455]]
27:
 [[1432 5989]
 [  72  507]]
28:
 [[1127 6294]
 [  50  529]]
29:
 [[ 856 6565]
 [  30  549]]
30:
 [[2695 4726]
 [ 141  438]]
31:
 [[2010 5411]


In [20]:
beta2_params_dt_fcw_op[strvar] = study.best_params
#weighting=beta2_params_dt_fcw_op[strvar]['class_weight_opt']
#print(weighting)
#beta2_params_dt_fcw_op[strvar]['class_weight']=cw_map[weighting]
beta2_matrix_dt_fcw_op[strvar] = confusion_matrix(y_test, pred)

In [21]:
for key in beta2_params_dt_fcw_op:
    print(key, "\t:", beta2_params_dt_fcw_op[key],'\n')

one 	: {'class_weight_opt': 31, 'ccp_alpha': 0.00030320959037639743, 'max_depth': 28, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2', 'splitter': 'best', 'criterion': 'log_loss'} 



In [22]:
#%store -r
#print(beta2_params_dt_fcw_op['one'])

In [23]:
for key in beta2_matrix_dt_fcw_op:
    print(key, ":")
    for row in beta2_matrix_dt_fcw_op[key]:
        print(row)#for element in row:
            #print(element)
    print('\n')

one :
[2010 5411]
[120 459]




In [24]:
precision, recall, fbeta, support = precision_recall_fscore_support(
    y_test,
    pred,
    beta=beta2[strvar]
)

for i in range(len(precision)):
    print(f"beta value: {beta2[strvar]}")
    print(f"Class {i}")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  Fbeta:   {fbeta[i]:.4f}")
    print(f"  Support:   {support[i]}")

accuracy = accuracy_score(y_test, pred)

print(f"Accuracy: {accuracy:.4f}")

beta value: 1.0
Class 0
  Precision: 0.9437
  Recall:    0.2709
  Fbeta:   0.4209
  Support:   7421
beta value: 1.0
Class 1
  Precision: 0.0782
  Recall:    0.7927
  Fbeta:   0.1423
  Support:   579
Accuracy: 0.3086


In [25]:
#precision, recall, fbeta, support = precision_recall_fscore_support(
#    y_test,
#    pred,
#    beta=beta2[strvar]
#)
beta2_scores_dt_fcw_op[strvar] = {'precision0':-1}
for i in range(len(precision)):
    #print(f"beta value: {beta2[strvar]}")
    #print(f"Class {i}")
    #print(f"  Precision: {precision[i]:.4f}")
    if i == 0: 
        beta2_scores_dt_fcw_op[strvar]['precision0'] = precision[i]; beta2_scores_dt_fcw_op[strvar]['recall0'] = recall[i]; 
        beta2_scores_dt_fcw_op[strvar]['fbeta0'] = fbeta[i]
    if i == 1: 
        beta2_scores_dt_fcw_op[strvar]['precision1'] = precision[i]; beta2_scores_dt_fcw_op[strvar]['recall1'] = recall[i];
        beta2_scores_dt_fcw_op[strvar]['fbeta1'] = fbeta[i]
    #print(f"  Recall:    {recall[i]:.4f}")
    #print(f"  Fbeta:   {fbeta[i]:.4f}")
    #print(f"  Support:   {support[i]}")

accuracy = accuracy_score(y_test, pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.3086


In [26]:
for label in [0, 1]:
    score = fbeta_score(
        y_test,
        pred,
        beta=beta2[strvar],
        pos_label=label
    )
    print(f"Class {label} Fbeta-score: {score:.4f}")

print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
#print(f"dict of dict: ", beta2_scores_dt_fcw_op)

Class 0 Fbeta-score: 0.4209
Class 1 Fbeta-score: 0.1423
Accuracy: 0.3086


In [27]:
for key in beta2_scores_dt_fcw_op:
    print(key, ':', beta2_scores_dt_fcw_op[key], '\n')

one : {'precision0': np.float64(0.9436619718309859), 'recall0': np.float64(0.2708529847729417), 'fbeta0': np.float64(0.4208983352528531), 'precision1': np.float64(0.07819420783645656), 'recall1': np.float64(0.7927461139896373), 'fbeta1': np.float64(0.14234765079857342)} 



In [28]:
#%store beta2_params_dt_fcw_op
#%store beta2_matrix_dt_fcw_op
#%store beta2_scores_dt_fcw_op